# Определение параметра модели SOCS для измеренных точек

https://docs.google.com/document/d/1QBmiptBWSGtyMtSd0oLZUQnVQVi2s-Ts/edit

In [1]:
import os
import pandas as pd

### Данные измерений

В файле _prod.csv_ хранятся результаты измерений.

Поля:

- N, E - координаты, градусы
- organic - содержание органического вещества, Кг/м^2/мес
- group - veg/soil
- province - провинция
- с - содержание углерода в почве, Кг/м^2/мес
- Cm - коэфф. SOCS модели, максимальное количество органического углерода, которое может быть защищено в почве (см. background)
- fraction - процентное содержание фракции ила и песка в почве, %
- bd - плотность почвы, Кг/м^3

In [2]:
base_path = 'data'

In [3]:
from data_process.obs_data_reader import read_prod_data

soil_data_path = os.path.join(base_path, 'prod_c.csv')
soil_data = read_prod_data(soil_data_path, only_soil=True)

In [5]:
soil_data[['lat', 'lon']].to_csv(os.path.join(base_path, 'obs_coordinates.csv'), index=False, header=False)

### Форсинг

Внешний форсинг (температура, влажность почвы) из ERA5 в файле _tsoil_wsoil_era5.nc_.

Данные в точках измерения почвы в течение 11 лет с шагом месяц. Берется среднее значение.


In [6]:
from data_process.forcing_reader import read_forcing_at_sites

forcing_data = read_forcing_at_sites(os.path.join(base_path, 'tsoil_wsoil_era5.nc'))

In [7]:
df = pd.merge(soil_data, forcing_data, how='left', on=['lat', 'lon'])
df.sort_values(by=['lat', 'lon'], inplace=True)

print('Количество станций:', len(df))
df.head()

Количество станций: 123


,lat,lon,organic,group,province,veg_type,c,Cm,fraction,bd,Tsoil,Wsoil
0,43.1,44.6,0.079286,soil,М1,Сельхоз,16.366179,17.741997,58.10,1.170000,284.001617,0.347651
1,43.3,43.2,0.079286,soil,М1,Сельхоз,17.223462,24.965377,91.15,1.049400,277.574860,0.381184
2,43.3,44.3,0.026537,soil,Л1,Лес,6.230363,28.405732,93.50,1.164002,284.532227,0.323967
3,43.4,44.9,0.082167,soil,Н1,Трава,8.374185,22.719528,93.60,0.930000,285.639496,0.274306
4,43.4,45.0,0.030000,soil,О2,Сельхоз,9.145812,30.006230,88.80,1.294667,285.747101,0.279532


Оптимальные значения параметра kd для этих станций.

Получены в результате 30 000 итераций монте-карло.

In [8]:
kd_opt = pd.read_csv(os.path.join(base_path, 'kd_ref.csv'), header=None)[0].to_numpy()

df['kd_opt'] = kd_opt

In [9]:
df

,lat,lon,organic,group,province,veg_type,c,Cm,fraction,bd,Tsoil,Wsoil,kd_opt
0,43.1,44.6,0.079286,soil,М1,Сельхоз,16.366179,17.741997,58.100000,1.170000,284.001617,0.347651,0.056898
1,43.3,43.2,0.079286,soil,М1,Сельхоз,17.223462,24.965377,91.150000,1.049400,277.574860,0.381184,0.066709
2,43.3,44.3,0.026537,soil,Л1,Лес,6.230363,28.405732,93.500000,1.164002,284.532227,0.323967,0.079470
3,43.4,44.9,0.082167,soil,Н1,Трава,8.374185,22.719528,93.600000,0.930000,285.639496,0.274306,0.187695
4,43.4,45.0,0.030000,soil,О2,Сельхоз,9.145812,30.006230,88.800000,1.294667,285.747101,0.279532,0.057477
...,...,...,...,...,...,...,...,...,...,...,...,...,...
118,60.7,47.3,0.037820,soil,Е2,Лес,1.708458,27.816612,79.347826,1.343163,278.911194,0.345141,0.669734
119,60.7,47.4,0.037820,soil,Е2,Сельхоз,4.817979,24.414590,76.746667,1.218848,278.929199,0.344709,0.161459
120,60.8,56.9,0.037820,soil,Е2,NaN,4.363736,19.908613,62.000000,1.230294,278.171783,0.342376,0.185803
121,61.5,40.4,0.037820,soil,Е2,Лес,3.017074,14.855720,42.676667,1.333714,278.455597,0.309513,0.300791


#### Замечения

- опад (organic) берется как среднее по провинции
- температура и влажность почвы как среднее за 11 лет